In [ ]:
pip install modern_robotics

In [4]:
# Libraries
import modern_robotics as mr
import numpy as np
import sympy as sym
import matplotlib.pyplot as plt
from pprint import pprint
from IPython.display import Math

In [5]:
from IPython.display import HTML, display
display(HTML("""<table style="margin-left:auto; margin-right:auto;"> <tr><td><img src="https://i.ibb.co/4LQ0TbC/kinematics1.jpg" width="600" height="300"></td></tr></table>"""))

""


In [7]:
# Joint 1 : PIP
# Joint 2: MCP
# Joint 3: Splay

'''
r_pulley_motor_PIP = 3.0
r_pulley_motor_MCP = 4.0
r_pulley_motor_splay = 2.0

y_offset_from_bent_finger = 3.0
x_offset_b_frame_to_PIP = 4.0
x_offset_PIP_to_MCP = 3.0

Motor_Radii_Matrix = 0
'''

a, b, c, d = sym.symbols(r'a, b, c, d') # finger dimensions
rj, rw, rs = sym.symbols(r'rj, rw, rs') # pulley radii (tendons)
Rm1, Rm2, Rm3 = sym.symbols(r'Rm1, Rm2, Rm3') # pulley radii (motors)
th1, th2, th3 = sym.symbols(r'th1, th2, th3') # joint angles
f_tip_ext = sym.symbols(r'f_tip_ext') # force at finger tip extension
f_tip_splay = sym.symbols(r'f_tip_splay') # force at finger tip splay

In [8]:
# Matrices

# Blist
b1 = sym.Matrix([0, 0, 1, d, -a, 0])
b2 = sym.Matrix([0, 0, 1, d, -a-b, 0])
b3 = sym.Matrix([0, 1, 0, 0, 0, a+b+c])
B_list = sym.Matrix.hstack(b1, b2, b3)
display(Math(r"B=" + sym.latex(sym.Matrix(B_list))))
print("\n")

# Thetalist
theta_list = sym.Matrix([th1, th2, th3]).T
display(Math(r"theta\_list =" + sym.latex(sym.Matrix(theta_list))))
print("\n")

# Structure matrix
S = sym.Matrix([[rj, 0], [-rj, rj], [rw, -rw]])
display(Math(r"S =" + sym.latex(sym.Matrix(S))))
print("\n")

# Max wrench
Wrench = sym.Matrix([0, 0, 0, 0, f_tip_ext, f_tip_splay])
display(Math(r"F_{tip} =" + sym.latex(sym.Matrix(Wrench))))
print("\n")

# Joint pulley radii
RJ = sym.Matrix([[rj, rj, rw]])
display(Math(r"RJ =" + sym.latex(sym.Matrix(RJ))))
print("\n")

# Motor radii matrix
RM_mat = sym.diag(Rm1, Rm2, Rm3)
display(Math(r"RM_{mat} =" + sym.latex(sym.Matrix(RM_mat))))
print("\n")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
# Compute torques given wrench
def compute_joint_torques(Blist, theta_list, Wrench):
  Jb = mr.JacobianBody(Blist, theta_list)
  Tau_joints = Jb.T @ Wrench
  print("\n")
  display(Math(r"J\_b =" + sym.latex(sym.Matrix(Jb.T))))
  print("\n")
  display(Math(r"\tau_{\theta}=" + sym.latex(sym.Matrix(Tau_joints))))
  tau_joints = Tau_joints.reshape(3, )
  # print(tau_joints.shape)

  return tau_joints

# Compute net torque at joints (add spring action)
def compute_net_torques(tau_joints, spring_torques):
  # print(tau_joints.shape)
  # print(spring_torques.T.shape)
  res = tau_joints - spring_torques
  # print(f"res = {res}")
  print("\n")
  display(Math(r"\tau_{\theta net}=" + sym.latex(sym.Matrix(res))))
  return res

# Compute tendon tensions
def compute_tendon_tensions(net_torques, S_mat):
  # print(f'torques shape = {net_torques.shape}\nS_mat_shape = {S_mat.shape}')
  S_mat_inv = np.linalg.pinv(S_mat)
  res1 = S_mat_inv @ net_torques
  print("\n")
  display(Math(r"T_{tendons} =" + sym.latex(sym.Matrix(res1))))
  return res1

def compute_spring_torques(theta_list, k_pip, k_mcp, l_rest_pip, l_rest_mcp, OD_pip, OD_mcp, angle_pip, angle_mcp, offset_pip, offset_mcp):
  # torque at theta1
  pip_spring_length = 2*offset_pip + theta_list[0]/(2*np.pi)*OD_pip*np.pi
  mcp_spring_length = 2*offset_mcp + theta_list[1]/(2*np.pi)*OD_mcp*np.pi
  pip_spring_x = pip_spring_length - l_rest_pip
  mcp_spring_x = mcp_spring_length - l_rest_mcp
  F_pip = k_pip*pip_spring_x
  F_mcp = k_mcp*mcp_spring_x
  tau_pip = F_pip*np.cos(angle_pip)
  tau_mcp = F_mcp*np.cos(angle_mcp)
  return np.array([tau_pip, tau_mcp,0])


def compute_straight_spring(rest_spring_length, cur_spring_length, k, width_to_spring):
  F = k*(cur_spring_length - rest_spring_length)
  tau = F*width_to_spring
  return tau

def compute_joint_speed_from_motor_speed(motor_speed, pulley_radii_motor, S_mat):
  motor_speed_rad_s = motor_speed*2*np.pi/60
  tendon_speed = (pulley_radii_motor @ motor_speed_rad_s)[0:2]
  S_mat_t = S_mat.T
  S_mat_t_inv = np.linalg.pinv(S_mat_t)
  joint_speed = S_mat_t_inv @ tendon_speed
  display(Math(r"\Pulley radii =" + sym.latex(sym.Matrix(pulley_radii_motor))))
  display(Math(r"\Smat_inv =" + sym.latex(sym.Matrix(S_mat_t_inv))))
  display(Math(r"\Joint speed =" + sym.latex(sym.Matrix(joint_speed))))
  return joint_speed




# Compute motor torques
def compute_motor_torques(Joint_torques, Tendon_t, pulley_radii_joints, pulley_radii_motor, radius_belt_splay):
  tau_th3_tendons = Tendon_t[1]*pulley_radii_joints[2] - Tendon_t[0]*pulley_radii_joints[2]
  tau_th3_jacob = Joint_torques[2]
  tau_belt = tau_th3_tendons + tau_th3_jacob

  tau_m12 = pulley_radii_motor @ np.array([Tendon_t[0], Tendon_t[1], 0]) # last value discarded
  tau_m3 = pulley_radii_motor[2][2]/radius_belt_splay * tau_belt

  # print(f'debug: {tau_m12}, {tau_m3}, {radius_belt_splay}\n')
  motor_torques = np.array([tau_m12[0], tau_m12[1], tau_m3])
  print("\n")
  display(Math(r"\tau_{motors} =" + sym.latex(sym.Matrix(motor_torques))))
  return motor_torques

In [10]:
# Given 20N force at finger tip
values = {f_tip_ext:20, f_tip_splay: 0, rj:0.008, rw:0.012, Rm1:0.01, Rm2:0.01, Rm3:0.01, a:0.0521, b:0.053, c:0.0299, d:0.007, th1:0, th2:0, th3:0}

B_list1 = np.array(B_list.subs(values)).astype(float)
theta_list1 = np.array(theta_list.subs(values)).astype(float).flatten()
S1 = np.array(S.subs(values)).astype(float)
W1 = np.array(Wrench.subs(values)).astype(float)
k_spring = 35.025 # N/m
rest_length = 0.018 # m
extended_length_pip = 0.0323 # m
width_to_spring = 0.0115 # m
extended_length_mcp = 0.0323 # m FIX!!!!!!!!!!
spring_torques = np.array([compute_straight_spring(rest_length, extended_length_pip, k_spring, width_to_spring), compute_straight_spring(rest_length, extended_length_mcp, k_spring, width_to_spring), 0])

# spring_torques = compute_spring_torques(theta_list, k_pip, k_mcp, l_rest_pip, l_rest_mcp, OD_pip, OD_mcp, angle_pip, angle_mcp, offset_pip, offset_mcp)
# Flatten RJ1 to make it a 1D array as expected for direct indexing of radii
RJ1 = np.array(RJ.subs(values)).astype(float).flatten()
RM1 = np.array(RM_mat.subs(values)).astype(float)
rs = 0.0127

Jt = compute_joint_torques(B_list1, theta_list1, W1)
Jt_net = compute_net_torques(Jt, spring_torques)
T_vec = compute_tendon_tensions(Jt_net, S1)
tau_motors = compute_motor_torques(Jt_net, T_vec, RJ1, RM1, rs)
print(theta_list1)
'''
print(B_list1)
print(theta_list1)
print(S1)
print(W1)
'''

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[0. 0. 0.]


'\nprint(B_list1)\nprint(theta_list1)\nprint(S1)\nprint(W1)\n'

In [11]:
# Given 10 N force at splay
splay_values = {f_tip_ext:0, f_tip_splay: 10, rj:0.005, rw:0.012, Rm1:0.01, Rm2:0.01, Rm3:0.01, a:0.0521, b:0.053, c:0.0299, d:0.007, th1:0, th2:0, th3:0}

B_list1 = np.array(B_list.subs(splay_values)).astype(float)
theta_list1 = np.array(theta_list.subs(splay_values)).astype(float).flatten()
S1 = np.array(S.subs(splay_values)).astype(float)
W1 = np.array(Wrench.subs(splay_values)).astype(float)
k_spring = 35.025 # N/m
rest_length = 0.018 # m
extended_length_pip = 0.0323 # m
width_to_spring = 0.0115 # m
extended_length_mcp = 0.0323 # m FIX!!!!!!!!!!
spring_torques = np.array([compute_straight_spring(rest_length, extended_length_pip, k_spring, width_to_spring), compute_straight_spring(rest_length, extended_length_mcp, k_spring, width_to_spring), 0])
# spring_torques = compute_spring_torques(theta_list, k_pip, k_mcp, l_rest_pip, l_rest_mcp, OD_pip, OD_mcp, angle_pip, angle_mcp, offset_pip, offset_mcp)
# Flatten RJ1 to make it a 1D array as expected for direct indexing of radii
RJ1 = np.array(RJ.subs(splay_values)).astype(float).flatten()
RM1 = np.array(RM_mat.subs(splay_values)).astype(float)
rs = 0.0127

Jt = compute_joint_torques(B_list1, theta_list1, W1)
Jt_net = compute_net_torques(Jt, spring_torques)
T_vec = compute_tendon_tensions(Jt_net, S1)
tau_motors = compute_motor_torques(Jt_net, T_vec, RJ1, RM1, rs)
print(theta_list1)
print((0.0521+0.053+0.0299+0.007)*10)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[0. 0. 0.]
1.4200000000000002


In [12]:
#   compute_joint_speed_from_motor_speed(np.array([10, 10, 0]), RM1, S1)